# Rank Order Project Plan
In-Class Assignment #1
Test by Mina Jung

## Draft 1
Next steps:
+ Test input parameters.
  + have the code reusable?
  + test with many data
+ Measure efficiency. theory, time.perf_counter
+ Compare Objectives: pareto optimal, popular, or profile-based ?
+ Compare mathematical implementations (from our literature).
+ See if these can be incorporated to improve efficiency.
+ readme (ie. code document)
  + model assumptions
  + design decisions
  + references
  + algorithm explanation
+ baseline
+ team contributions


Imports

In [10]:
import random
from time import time
random.seed(42)

Variable name definitions:
+ R = resident preferences
+ C = hospital capacities
+ A_arr = resident assignments
+ H = applicants in the current instance

Initialize inputs

In [11]:
# Input: Resident preferences
# R in numpy array form
##R = [[1,2,3],
##     [1,2,3],
##   [1,3,2],
##     [3,1,2]]

# Original capacity
##C = [2,1,1]

"""
Generate large, valid R / C / H inputs for the RGS algorithm.

Matches the data conventions already used in the notebook:
    R : list of lists, 1-indexed hospital numbers (R[doctor] = ranked list)
    C : list of ints, hospital capacities (0-indexed by hospital position)
    H : list of empty lists, one per hospital, ready for round 1 of RGS

Guarantees enforced:
    1. sum(C) >= n_doctors  (total hospital capacity covers every doctor)
    2. every hospital appears on every doctor's list exactly once
       (each R[d] is a full permutation of 1..n_hospitals, so no two
       hospitals share the same rank on any doctor's list)
"""

import random


def generate_large_input(n_doctors, n_hospitals, seed=None,
                          min_capacity=1, max_extra_capacity=3):
    """
    Parameters
    ----------
    n_doctors : int
        Number of residents/doctors to generate preferences for.
    n_hospitals : int
        Number of hospitals to generate capacities for.
    seed : int, optional
        RNG seed for reproducibility.
    min_capacity : int
        Capacity every hospital is guaranteed to start with, so no
        hospital is randomly left at 0 seats.
    max_extra_capacity : int
        Upper bound on extra random slack added per hospital on top of
        the minimum + doctor-covering allocation, so total capacity
        comfortably exceeds n_doctors rather than just equaling it.

    Returns
    -------
    R : list[list[int]]
        R[d] is doctor d's full ranked preference list over hospitals,
        1-indexed (1..n_hospitals), each hospital appearing exactly once.
    C : list[int]
        C[h] is hospital h's capacity.
    H : list[list[int]]
        Empty per-hospital applicant buckets: H[h] = [] for every hospital,
        the shape RGS expects at the start of round 1.
    """
    rng = random.Random(seed)

    # --- R: every doctor ranks every hospital exactly once ---
    hospitals = list(range(1, n_hospitals + 1))  # 1-indexed, matches notebook
    R = []
    for _ in range(n_doctors):
        prefs = hospitals[:]
        rng.shuffle(prefs)
        R.append(prefs)

    # --- C: random capacities guaranteed to sum to >= n_doctors ---
    C = [min_capacity] * n_hospitals
    remaining_needed = max(0, n_doctors - sum(C))
    for _ in range(remaining_needed):
        C[rng.randrange(n_hospitals)] += 1
    for h in range(n_hospitals):
        C[h] += rng.randint(0, max_extra_capacity)

    # --- H: empty applicant buckets, one per hospital ---
    H = [[] for _ in range(n_hospitals)]

    return R, C, H


def validate_input(R, C, n_doctors, n_hospitals):
    """Sanity-check the assumptions the generator is supposed to guarantee."""
    assert sum(C) >= n_doctors, "Total capacity is less than the number of doctors"
    for d, prefs in enumerate(R):
        assert len(prefs) == n_hospitals, f"Doctor {d} preference list has wrong length"
        assert sorted(prefs) == list(range(1, n_hospitals + 1)), (
            f"Doctor {d} preference list is missing a hospital or has a duplicate rank"
        )


RGS Algorithm

In [12]:
### Subsequent Rounds

def RGS(R, C, test_mode):
    if sum(C) < len(R):
        print("Please read the README and make sure the number of doctors and hospitals are correct")
        return 
    # Convert to 0-based indexing
    R = [[i-1 for i in r] for r in R]
    A_arr = [None] * len(R)
    # Remaining capacity
    remaining_C = C.copy()

    pref_round = 0
    current_applicants = list(range(len(R)))

    # Implicit way to run loop while current_applicants is NOT empty
    # We also assume that each resident's preference list is the same, hence why taking len(r1) would be sufficient
    while current_applicants and (pref_round<len(R[0])):

        # New H for this round
        H = [[] for _ in range(len(C))]

        # Build H from current applicants
        for resident, choices in enumerate(R):
            if resident in current_applicants:
                H[choices[pref_round]].append(resident)

        if test_mode:
           print(f"Hospital choices for round {pref_round+1}: ", end="")
           print_H(H)

        # Accept or reject applicants
        round_reject = []
        for hospital, applicants in enumerate(H):
            if len(applicants) > remaining_C[hospital]: # Too many applicants per capacity
                accepted = random.sample(applicants, remaining_C[hospital])
                rejected = [item for item in applicants if item not in accepted]
                round_reject += rejected
                if test_mode: print_acc_rej(hospital, accepted, rejected)
            else:
                accepted = applicants
                rejected = []
                if test_mode: print_acc_rej(hospital, accepted, rejected)
            for resident in accepted: # Separating consequences of decision
                A_arr[resident] = hospital
                remaining_C[hospital]-= 1

        if test_mode:
          print(f"Assignments for round {pref_round+1}: ", end="")
          print_A(A_arr)
          print(f"Rejections for round {pref_round+1}: ", end="")
          print_app(round_reject, end="\n")
          print(f"Remaining capacities for round {pref_round+1}: ", end="")
          print_C(remaining_C)

        # Update round
        current_applicants = round_reject
        pref_round += 1

    if test_mode:
        print(f"Final assignments: ", end="")
        print_A(A_arr)
        print(f"Final capacities: ", end="")
        print_C(remaining_C)

    return A_arr

In [13]:
### Subsequent Rounds

def Random(R, C, test_mode):
    if sum(C) < len(R):
        print("Please read the README and make sure the number of doctors and hospitals are correct")
        return None
    A_arrR = [None] * len(R)
    remaining_C = C.copy()

    residents = list(range(len(R)))
    random.shuffle(residents)

    hospitals_with_room = [h for h in range(len(C)) if remaining_C[h] > 0]
    for resident in residents:
        if not hospitals_with_room:
            if test_mode:
                print(f"Resident {resident} could not be placed — no capacity left anywhere")
            continue
        hospital = random.choice(hospitals_with_room)
        A_arrR[resident] = hospital
        remaining_C[hospital] -= 1
        if remaining_C[hospital] == 0:
            hospitals_with_room.remove(hospital)

    return A_arrR

Print helper functions

In [14]:
def print_acc_rej(hospital, accepted, rejected):
    print(f"'h{hospital + 1}': Accepted: ", end="")
    print_app(accepted, end=" ")
    print(f"Rejected: ", end="")
    print_app(rejected, end="\n")

def print_A(A):
    print("{", end="")
    for resident, hospitals in enumerate(A):
        print(f"'r{resident + 1}': ", end="")
        print(f"'h{hospitals}', ", end="")
    print("}")

def print_H(H):
    print("{", end="")
    for hospital, applicants in enumerate(H):
        print(f"'h{hospital + 1}': ", end="")
        print_app(applicants)
        print(", ", end="")
    print("}")

def print_C(C):
    print("{", end="")
    for hospital, capacity in enumerate(C):
        print(f"'h{hospital + 1}': {capacity}, ", end="")
    print("}")

def print_hosp(h_s):
    print(f"{[f'h{h + 1}' for h in h_s]}", end="")

def print_app(r_s, end="false"):
    print(f"{[f'r{r + 1}' for r in r_s]}", end="")
    if end != "false":
        print(end, end="")



def averageRank(R, A_arr):
    if A_arr is None:
        print("No assignment possible")
        return float("nan")
    A_Rank = [R[resident].index(A_arr[resident] + 1) + 1 for resident in range(len(R))]
    averageRrank = sum(A_Rank) / len(A_Rank)
    print("Average Rank:", averageRrank)
    return averageRrank

def testing(R,C):
    milliS = int(time() * 1000)*1000
    A_arr = RGS(R, C, test_mode=True)
    milliEnd = int(time() * 1000)*1000
    ARTime = milliEnd-milliS
    print("Time for Algorithm:", ARTime ,"microseconds")
    print("   ")
       
        
    milliRandomStart = int(time() * 1000)*1000
    A_arrR = Random(R, C, test_mode=True)
        
    milliRandomEnd = int(time() * 1000)*1000
    RTime = milliRandomEnd-milliRandomStart
    print("   ")
    print("   ")
    print("   ")
    print("   ")
    averageRank(R,A_arr)
    print("Time for Algorithm:", ARTime ,"milliseconds")
    print("   ")
    averageRank(R,A_arrR)
    print("Time for Random Algorithm:", RTime ,"milliseconds")


Test algorithm on inputs

In [19]:
if __name__ == "__main__":
    n_doctors = 5
    n_hospitals = 3

    R, C, H = generate_large_input(n_doctors, n_hospitals, seed=42)
    
    totalC = sum(C)
    print(f"Doctors: {n_doctors}, Hospitals: {n_hospitals}")
    print(f"Total capacity: {totalC} (needs >= {n_doctors})")
    print(f"Capacities (C): {C}")
    print(f"Sample doctor 0 preference list (R[0]): {R[0]}")
    print(f"Initial H (empty applicant buckets): {H}")
    print()

    validate_input(R, C, n_doctors, n_hospitals)
    testing(R,C)
    print("    ")
    print("    ")
    print("    ")
    print("    ")
# ====================================================================
# Test 0: Input does not meet assumptions: total capacity < number of residents

    R = [[3,2,1],
         [3,2,1],
         [3,2,1],
         [3,2,1]]
    
    C = [1,1,1]
    
    print("Test 0")
    testing(R,C)
    print("    ")
    print("    ")
    print("    ")
    print("    ")
    # ====================================================================
    # Test 1: No residents have conflicting preferences
    
    # Resident preferences
    R = [[1,2,3],
         [3,1,2],
         [2,3,1]]
    
    # Original hospital capacity
    C = [1,1,1]
    
    print("Test 1")
    testing(R,C)
    print("    ")
    print("    ")
    print("    ")
    print("    ")
    # ====================================================================
    # Test 2: Some residents have conflicting preferences
    
    # Resident preferences
    R = [[1,2,3],
         [1,3,2],
         [2,3,1]]
    
    # Original hospital capacity
    C = [1,1,1]
    print("Test 2")
    testing(R,C)
    print("    ")
    print("    ")
    print("    ")
    print("    ")
    # ====================================================================
    # Test 3: More residents than number of hospitals
    
    # Resident preferences
    R = [[1,2,3],
         [1,2,3],
         [1,3,2],
         [3,1,2]]
    
    # Original hospital capacity
    C = [2,1,1]
    print("Test 3")
    testing(R,C)
    print("    ")
    print("    ")
    print("    ")
    print("    ")
    # ====================================================================
    # Test 4: Less residents than total capacity
    
    R = [[3,2,1,4],
         [3,2,4,1],
         [2,1,4,3]]
    
    C = [1,2,1]
    print("Test 4")
    testing(R,C)
    print("    ")
    print("    ")
    print("    ")
    print("    ")
    # ====================================================================
    # Test 5: Hospital with 0 capacity
    
    # Resident preferences
    R = [[1,2,3],
         [3,1,2],
         [2,3,1]]
    
    # Original hospital capacity
    C = [1,0,2]
    print("Test 5")
    testing(R,C)
    print("    ")
    print("    ")
    print("    ")
    print("    ")
    # ====================================================================
    # Test 6: Large dataset
    
    #R = [[]]
    
    #C = []
    
    # ====================================================================
    

   

Doctors: 5, Hospitals: 3
Total capacity: 5 (needs >= 5)
Capacities (C): [1, 2, 2]
Sample doctor 0 preference list (R[0]): [2, 1, 3]
Initial H (empty applicant buckets): [[], [], []]

Hospital choices for round 1: {'h1': [], 'h2': ['r1', 'r3', 'r4', 'r5'], 'h3': ['r2'], }
'h1': Accepted: [] Rejected: []
'h2': Accepted: ['r4', 'r5'] Rejected: ['r1', 'r3']
'h3': Accepted: ['r2'] Rejected: []
Assignments for round 1: {'r1': 'hNone', 'r2': 'h2', 'r3': 'hNone', 'r4': 'h1', 'r5': 'h1', }
Rejections for round 1: ['r1', 'r3']
Remaining capacities for round 1: {'h1': 1, 'h2': 0, 'h3': 1, }
Hospital choices for round 2: {'h1': ['r1'], 'h2': [], 'h3': ['r3'], }
'h1': Accepted: ['r1'] Rejected: []
'h2': Accepted: [] Rejected: []
'h3': Accepted: ['r3'] Rejected: []
Assignments for round 2: {'r1': 'h0', 'r2': 'h2', 'r3': 'h2', 'r4': 'h1', 'r5': 'h1', }
Rejections for round 2: []
Remaining capacities for round 2: {'h1': 0, 'h2': 0, 'h3': 0, }
Final assignments: {'r1': 'h0', 'r2': 'h2', 'r3': 'h2', 'r4

## Archive

In [16]:
# Input: Resident preferences
r1 = ["h1", "h2", "h3"]
r2 = ["h1", "h2", "h3"]
r3 = ["h1", "h3", "h2"]
r4 = ["h3", "h1", "h2"]

# R in dictionary form
R = {
    "r1": r1,
    "r2": r2,
    "r3": r3,
    "r4": r4,
}

# Hospital POV on applicants
H = {
    "h1": [],
    "h2": [],
    "h3": []
}

# c_h1 = 1; c_h2 = 2; c_h3 = 1 # capacity of hotels
# Capacity in dictionary form
C = {
    "h1": 1,
    "h2": 2,
    "h3": 1
}

# pref_round = 0
# for resident, choices in R.items():
#     if choices[pref_round] == "h1":
#         H["h1"].append(resident)
#     elif choices[pref_round] == "h2":
#         H["h2"].append(resident)
#     elif choices[pref_round] == "h3":
#         H["h3"].append(resident)
pref_round = 0
for resident, choices in R.items():
    H[choices[pref_round]].append(resident)
print(H)

for hospital, applicants in H.items():
    n_applicants = len(applicants)
    n_capacity = C[hospital]
    print(f"{hospital}: {n_applicants} applicants, capacity {n_capacity}")

# h1 only: We randomly accept one applicant (capacity) and reject the remaining two applicants.
accepted = random.sample(H["h1"], C["h1"])
# for resident in accepted:
    # H["h1"].remove(resident)
# rejected = H["h1"]
rejected = [item for item in H["h1"] if item not in accepted] # Python list comprehension if not in another list
print(f"Accepted: {accepted} Rejected: {rejected}")

# h1 only: We compare applicants with capacity.
if len(H["h1"]) > C["h1"]:
    accepted = random.sample(H["h1"], C["h1"])
    rejected = [item for item in H["h1"] if item not in accepted]
    print(f"Accepted: {accepted} Rejected: {rejected}")
else:
    accepted = H["h1"]
    rejected = []
    print(f"Accepted: {accepted} Rejected: {rejected}")

{'h1': ['r1', 'r2', 'r3'], 'h2': [], 'h3': ['r4']}
h1: 3 applicants, capacity 1
h2: 0 applicants, capacity 2
h3: 1 applicants, capacity 1
Accepted: ['r3'] Rejected: ['r1', 'r2']
Accepted: ['r2'] Rejected: ['r1', 'r3']


In [17]:
for hospital, applicants in H.items():
    if len(applicants) > C[hospital]:
        accepted = random.sample(applicants, C[hospital])
        rejected = [item for item in applicants if item not in accepted]
        print(f"{hospital}: Accepted: {accepted} Rejected: {rejected}")
    else:
        accepted = applicants
        rejected = []
        print(f"{hospital}: Accepted: {accepted} Rejected: {rejected}")

h1: Accepted: ['r3'] Rejected: ['r1', 'r2']
h2: Accepted: [] Rejected: []
h3: Accepted: ['r4'] Rejected: []


In [18]:
# We carry over rejected applicants from the previous round to move on to their second choice.
pref_round = 1
H = {
    "h1": [],
    "h2": [],
    "h3": []
}
for resident, choices in R.items():
    if resident in round_reject:
        H[choices[pref_round]].append(resident)
print(H)

NameError: name 'round_reject' is not defined

In [ ]:
### Round 1

# Hospital POV on applicants
H = {
    "h1": [],
    "h2": [],
    "h3": []
}

# First choice, initialization of H
pref_round = 0
for resident, choices in R.items():
    H[choices[pref_round]].append(resident)
print(f"Initial hospital choices: {H}")

# Accept or reject applicants
round_reject = []
    for hospital, applicants in H.items():
        if len(applicants) > remaining_C[hospital]: # Too many applicants per capacity
            accepted = random.sample(applicants, remaining_C[hospital])
            rejected = [item for item in applicants if item not in accepted]
            round_reject += rejected
            print(f"{hospital}: Accepted: {accepted} Rejected: {rejected}")
        else:
            accepted = applicants
            rejected = []
            print(f"{hospital}: Accepted: {accepted} Rejected: {rejected}")
        for resident in accepted: # Separating consequences of decision
            A_dict[resident].append(hospital)
            remaining_C[hospital]-= 1

    print(f"Assignments for round 1: {A_dict}")
    print(f"Rejections for round 1: {round_reject}")
    print(f"Remaining capacities for round 1: {remaining_C}")

In [ ]:
# Trying to put things together

# Input: Resident preferences
r1 = ["h1", "h2", "h3"]
r2 = ["h1", "h2", "h3"]
r3 = ["h1", "h3", "h2"]
r4 = ["h3", "h1", "h2"]

# R in dictionary form
R = {
    "r1": r1,
    "r2": r2,
    "r3": r3,
    "r4": r4,
}


pref_round = 0

H = {
    "h1": [],
    "h2": [],
    "h3": []
}
C = {
    "h1": 1,
    "h2": 2,
    "h3": 1
}

In [ ]:
# Transform into dictionary?
choice_1 = []
choice_2 = []
choice_3 = []

for r_choice in R:
    choice_1.append(r_choice[0])
    choice_2.append(r_choice[1])
    choice_3.append(r_choice[2])

res_pref = {
    "r1": r1,
    "r2": r2,
    "r3": r3
}
print(res_pref)

rank_choice = {
    "choice_1": {
        "n_h1": [choice_1.count("h1"), c_h1],
        "n_h2": [choice_1.count("h2"), c_h2],
        "n_h3": [choice_1.count("h3"), c_h3]
    },
    "choice_2": {
        "n_h1": [choice_2.count("h1"), c_h1],
        "n_h2": [choice_2.count("h2"), c_h2],
        "n_h3": [choice_2.count("h3"), c_h3]
    },
    "choice_3": {
        "n_h1": [choice_3.count("h1"), c_h1],
        "n_h2": [choice_3.count("h2"), c_h2],
        "n_h3": [choice_3.count("h3"), c_h3]
    }
}

print(rank_choice)

In [ ]:
rank_result =

for choices, counts in rank_choice.items():
    for key, value in counts.items():
        print(key, value)
        if value[0] > value[1]:
            # move residents to next round
            # TBD
        elif (value[0] <= value[1]) and (value[0] != 0):
            # update hospital capacity
            if key == 'n_h1':
                c_h1 = value[1]-value[0]
            elif key == 'n_h2':
                c_h2 = value[1]-value[0]
            elif key == 'n_h3':
                c_h3 = value[1]-value[0]
            # assign resident
